1.BASELINE

In [1]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from types import MethodType
import json, pickle, os

c:\Users\TL1\anaconda3\envs\myexamenv\lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
c:\Users\TL1\anaconda3\envs\myexamenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODEL_NAME = "meta-llama/Llama-3.2-3B" 
TEST_FILE_PATH = "test_5.pkl"
ACTIVATION_MASK_PATH = "activation_mask.pth"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==========================================================
#  Load tokenizer & model
# ==========================================================
print("🔹 Loading model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None
)
model.eval()
print("✅ Model loaded.")

Using device: cuda
🔹 Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:08<00:00,  4.43s/it]


✅ Model loaded.


In [4]:
# ==========================================================
#  Load test data
# ==========================================================
print(f"🔹 Loading test data from {TEST_FILE_PATH}")
with open(TEST_FILE_PATH, "rb") as f:
    data = pickle.load(f)
test_token = data["inputs"]
labels = data["labels"]
print(f"✅ Loaded {len(test_token)} samples.")

🔹 Loading test data from test_5.pkl
✅ Loaded 60 samples.


In [5]:
print(f"🔹 Loading activation mask from {ACTIVATION_MASK_PATH}")
activation_masks = torch.load(ACTIVATION_MASK_PATH)
print("✅ Activation mask loaded.")


🔹 Loading activation mask from activation_mask.pth
✅ Activation mask loaded.


In [6]:
len(activation_masks[0])

28

In [7]:
# ==========================================================
#  Define inhibition hook for LLaMA-3.x MLPs
# ==========================================================
for activation_mask in activation_masks:
    if activation_mask is not None:

        def factory(mask):
            def llama_forward(self, x):
                gate = self.gate_proj(x)
                up = self.up_proj(x)
                activation = F.silu(gate)

                #mask_tensor = torch.ones(activation.size(-1), device=activation.device,dtype=activation.dtype)
                #mask_tensor[mask.to(activation.device).long()] = 0

                #activation = activation * mask_tensor
                x = activation * up
                x = self.down_proj(x)
                return x
            return llama_forward

        # Inject hook layer-wise
        for i, layer_mask in enumerate(activation_mask):
            obj = model.model.layers[i].mlp
            obj.forward = MethodType(factory(layer_mask.to(device).long()), obj)

print("✅ Mask hooks injected.")


✅ Mask hooks injected.


In [8]:
from tqdm import tqdm
preds=[]
print("\n--- Model Outputs ---")

# Find a good stop token ID. For Llama 3, <|end_of_turn|> is best.
try:
    stop_token_id = tokenizer.convert_tokens_to_ids("<|end_of_turn|>")
    if stop_token_id == tokenizer.unk_token_id:
        stop_token_id = tokenizer.eos_token_id
except:
    stop_token_id = tokenizer.eos_token_id
print(f"Using stop token ID: {stop_token_id}")

for i in tqdm(range(len(test_token)), desc="Generating Outputs"):
    input_ids = torch.tensor([test_token[i]], device=device)
    attention_mask = torch.ones_like(input_ids) # <-- FIX 1: Add attention mask
    label = labels[i].strip()

    # Generate output
    with torch.no_grad(): # Ensure no gradients are computed
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask, # <-- FIX 2: Pass attention mask
            max_new_tokens=100,
            do_sample=False, # <-- FIX 3: Add this for greedy decoding
            # temperature=0.0, # No longer needed when do_sample=False
            # top_p=1.0,       # No longer needed when do_sample=False
            eos_token_id=stop_token_id,
            pad_token_id=tokenizer.eos_token_id # Set pad_token_id
        )
    
    # Decode the *new* tokens only
    new_tokens = output_ids[0][len(test_token[i]):]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # --- 7. Display Results ---
    preds.append(prediction)
    print("\n---------------------------------")
    print(f"Example {i+1}")
    # prompt_text = tokenizer.decode(test_token[i], skip_special_tokens=True)
    # print(f"\nPROMPT:\n{prompt_text}") # Uncomment this line if you want to see the full prompt
    print(f"\nPREDICTION: {prediction}")
    print(f"LABEL:      {label}")


--- Model Outputs ---
Using stop token ID: 128001


Generating Outputs:   2%|▏         | 1/60 [00:03<03:42,  3.76s/it]


---------------------------------
Example 1

PREDICTION: वह कहते हैं कि मुख्य रूप से राजनीतिक दलों का कहना है कि आज समय को नहीं बढाया जाना चाहिए और कल मंत्री जवाब दे सकता है।.…

The minister is not available. นาง.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ.ﾆﾆ
LABEL:      उन्होंने यह भी कहा- ज्यादातर राजनीतिक पार्टियां ‘कह रही हैं कि वक्त आज नहीं बढ़ाया जाना चाहिए और मंत्री कल जवाब दे सकते हैं…।


Generating Outputs:   3%|▎         | 2/60 [00:07<03:24,  3.53s/it]


---------------------------------
Example 2

PREDICTION: क्योंकि भारत में... #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+#
LABEL:      दरअसल, भारतीय . पढ़ें


...


---------------------------------
Example 58

PREDICTION: जुलाई से सितंबर के तिमाही के लिए गपीएफ ने 7.6% की दर पर रिटर्न दिया।.…

The company has a turnover of 1.5 crore..…

The company has a turnover of 1.5 crore.…

The company has a turnover of 1.5 crore.…

The company has a turnover of 1.5 crore.…

The company has a turnover of 1.5
LABEL:      जुलाई-सितंबर तिमाही में जीपीएफ पर ब्‍याज 7.6 प्रतिशत वार्षिक था।


Generating Outputs:  98%|█████████▊| 59/60 [03:09<00:02,  2.73s/it]


---------------------------------
Example 59

PREDICTION: हरी परिवार को पता चला है कि हैरी और मेगन के लिए पिछले कुछ सालों में कितनी मुश्किलें आई हैं. +#+#+#+#+#+ +#+#+#+#+#+ +#+#+#+#+#+	"
LABEL:      बकिंघम पैलेस के बयान में कहा गया है कि पूरा परिवार यह जानकर दुखी है कि हैरी और मेगन के लिए पिछले कुछ वर्ष चुनौतीपूर्ण रहे हैं।


Generating Outputs: 100%|██████████| 60/60 [03:12<00:00,  3.21s/it]


---------------------------------
Example 60

PREDICTION: पुलिस ने शव और घायलों के परिवार के साथ संपर्क की कोशिश कर रही हैं।.…

The police are trying to contact the family members of the deceased and the injured..…

The police are trying to contact the family members of the deceased and the injured.…

The police are trying to contact the family members of the deceased and the injured.…

The police are trying to contact the family members of the deceased and the injured.…

The
LABEL:      वहीँ पुलिस मुर्तक और घायल लोगो के परिवार से संपर्क करने की कोशिश कर रही है.


In [9]:
from bert_score import score

# Compute BERTScore for all predictions vs labels
P, R, F1 = score(preds, labels, lang="hi")

# Print individual F1 scores
print("\n--- BERTScore for each example ---")
for i, f in enumerate(F1):
    print(f"Example {i+1}: {f.item():.4f}")

# Print average score
print("\nAverage BERTScore F1:", F1.mean().item())



--- BERTScore for each example ---
Example 1: 0.7531
Example 2: 0.4167
Example 3: 0.6950
Example 4: 0.6091
Example 5: 0.6722
Example 6: 0.7628
Example 7: 0.6663
Example 8: 0.5692
Example 9: 0.6521
Example 10: 0.4933
Example 11: 0.6591
Example 12: 0.6420
Example 13: 0.7204
Example 14: 0.7217
Example 15: 0.7840
Example 16: 0.6702
Example 17: 0.5777
Example 18: 0.6944
Example 19: 0.5803
Example 20: 0.6158
Example 21: 0.7439
Example 22: 0.6420
Example 23: 0.5938
Example 24: 0.7600
Example 25: 0.5685
Example 26: 0.6373
Example 27: 0.6266
Example 28: 0.7338
Example 29: 0.6662
Example 30: 0.7221
Example 31: 0.7485
Example 32: 0.6592
Example 33: 0.7154
Example 34: 0.7463
Example 35: 0.8621
Example 36: 0.6163
Example 37: 0.6036
Example 38: 0.6565
Example 39: 0.8496
Example 40: 0.6804
Example 41: 0.6296
Example 42: 0.6692
Example 43: 0.5840
Example 44: 0.7891
Example 45: 0.7034
Example 46: 0.6873
Example 47: 0.5810
Example 48: 0.6361
Example 49: 0.5756
Example 50: 0.5298
Example 51: 0.6534
Exam

2.AFTER DEACTIVATING THE IDENTIFIED NEURONS

In [7]:
# ==========================================================
#  Define inhibition hook for LLaMA-3.x MLPs
# ==========================================================
for activation_mask in activation_masks:
    if activation_mask is not None:

        def factory(mask):
            def llama_forward(self, x):
                gate = self.gate_proj(x)
                up = self.up_proj(x)
                activation = F.silu(gate)

                mask_tensor = torch.ones(activation.size(-1), device=activation.device,dtype=activation.dtype)
                mask_tensor[mask.to(activation.device).long()] = 0

                activation = activation * mask_tensor
                x = activation * up
                x = self.down_proj(x)
                return x
            return llama_forward

        # Inject hook layer-wise
        for i, layer_mask in enumerate(activation_mask):
            obj = model.model.layers[i].mlp
            obj.forward = MethodType(factory(layer_mask.to(device).long()), obj)

print("✅ Mask hooks injected.")


✅ Mask hooks injected.


In [8]:
from tqdm import tqdm
preds=[]
print("\n--- Model Outputs ---")

# Find a good stop token ID. For Llama 3, <|end_of_turn|> is best.
try:
    stop_token_id = tokenizer.convert_tokens_to_ids("<|end_of_turn|>")
    if stop_token_id == tokenizer.unk_token_id:
        stop_token_id = tokenizer.eos_token_id
except:
    stop_token_id = tokenizer.eos_token_id
print(f"Using stop token ID: {stop_token_id}")

for i in tqdm(range(len(test_token)), desc="Generating Outputs"):
    input_ids = torch.tensor([test_token[i]], device=device)
    attention_mask = torch.ones_like(input_ids) # <-- FIX 1: Add attention mask
    label = labels[i].strip()

    # Generate output
    with torch.no_grad(): # Ensure no gradients are computed
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask, # <-- FIX 2: Pass attention mask
            max_new_tokens=100,
            do_sample=False, # <-- FIX 3: Add this for greedy decoding
            # temperature=0.0, # No longer needed when do_sample=False
            # top_p=1.0,       # No longer needed when do_sample=False
            eos_token_id=stop_token_id,
            pad_token_id=tokenizer.eos_token_id # Set pad_token_id
        )
    
    # Decode the *new* tokens only
    new_tokens = output_ids[0][len(test_token[i]):]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # --- 7. Display Results ---
    preds.append(prediction)
    print("\n---------------------------------")
    print(f"Example {i+1}")
    # prompt_text = tokenizer.decode(test_token[i], skip_special_tokens=True)
    # print(f"\nPROMPT:\n{prompt_text}") # Uncomment this line if you want to see the full prompt
    print(f"\nPREDICTION: {prediction}")
    print(f"LABEL:      {label}")


--- Model Outputs ---
Using stop token ID: 128001


Generating Outputs:   2%|▏         | 1/60 [00:04<04:10,  4.25s/it]


---------------------------------
Example 1

PREDICTION: 
LABEL:      उन्होंने यह भी कहा- ज्यादातर राजनीतिक पार्टियां ‘कह रही हैं कि वक्त आज नहीं बढ़ाया जाना चाहिए और मंत्री कल जवाब दे सकते हैं…।


Generating Outputs:   3%|▎         | 2/60 [00:08<03:52,  4.01s/it]


---------------------------------
Example 2

PREDICTION: 
LABEL:      दरअसल, भारतीय . पढ़ें


...


---------------------------------
Example 58

PREDICTION: A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A A
LABEL:      जुलाई-सितंबर तिमाही में जीपीएफ पर ब्‍याज 7.6 प्रतिशत वार्षिक था।


Generating Outputs:  98%|█████████▊| 59/60 [04:05<00:04,  4.10s/it]


---------------------------------
Example 59

PREDICTION: 
LABEL:      बकिंघम पैलेस के बयान में कहा गया है कि पूरा परिवार यह जानकर दुखी है कि हैरी और मेगन के लिए पिछले कुछ वर्ष चुनौतीपूर्ण रहे हैं।


Generating Outputs: 100%|██████████| 60/60 [04:09<00:00,  4.16s/it]


---------------------------------
Example 60

PREDICTION: 
LABEL:      वहीँ पुलिस मुर्तक और घायल लोगो के परिवार से संपर्क करने की कोशिश कर रही है.


3.REMOVING ONLY THE SPECIFIC LAYERS

In [7]:
LAYER_NUMBER=3

In [8]:
# ==========================================================
#  Define inhibition hook for LLaMA-3.x MLPs
# ==========================================================
for activation_mask in activation_masks:
    if activation_mask is not None:

        def factory(mask):
            def llama_forward(self, x):
                gate = self.gate_proj(x)
                up = self.up_proj(x)
                activation = F.silu(gate)

                mask_tensor = torch.ones(activation.size(-1), device=activation.device,dtype=activation.dtype)
                mask_tensor[mask.to(activation.device).long()] = 0

                activation = activation * mask_tensor
                x = activation * up
                x = self.down_proj(x)
                return x
            return llama_forward

        # Inject hook layer-wise
        for i, layer_mask in enumerate(activation_mask):
            if i==3:
                obj = model.model.layers[i].mlp
                obj.forward = MethodType(factory(layer_mask.to(device).long()), obj)

            

print("✅ Mask hooks injected.")


✅ Mask hooks injected.


In [9]:
from tqdm import tqdm
preds=[]
print("\n--- Model Outputs ---")

# Find a good stop token ID. For Llama 3, <|end_of_turn|> is best.
try:
    stop_token_id = tokenizer.convert_tokens_to_ids("<|end_of_turn|>")
    if stop_token_id == tokenizer.unk_token_id:
        stop_token_id = tokenizer.eos_token_id
except:
    stop_token_id = tokenizer.eos_token_id
print(f"Using stop token ID: {stop_token_id}")

for i in tqdm(range(len(test_token)), desc="Generating Outputs"):
    input_ids = torch.tensor([test_token[i]], device=device)
    attention_mask = torch.ones_like(input_ids) # <-- FIX 1: Add attention mask
    label = labels[i].strip()

    # Generate output
    with torch.no_grad(): # Ensure no gradients are computed
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask, # <-- FIX 2: Pass attention mask
            max_new_tokens=100,
            do_sample=False, # <-- FIX 3: Add this for greedy decoding
            # temperature=0.0, # No longer needed when do_sample=False
            # top_p=1.0,       # No longer needed when do_sample=False
            eos_token_id=stop_token_id,
            pad_token_id=tokenizer.eos_token_id # Set pad_token_id
        )
    
    # Decode the *new* tokens only
    new_tokens = output_ids[0][len(test_token[i]):]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # --- 7. Display Results ---
    preds.append(prediction)
    print("\n---------------------------------")
    print(f"Example {i+1}")
    # prompt_text = tokenizer.decode(test_token[i], skip_special_tokens=True)
    # print(f"\nPROMPT:\n{prompt_text}") # Uncomment this line if you want to see the full prompt
    print(f"\nPREDICTION: {prediction}")
    print(f"LABEL:      {label}")


--- Model Outputs ---
Using stop token ID: 128001


Generating Outputs:   2%|▏         | 1/60 [00:03<03:37,  3.68s/it]


---------------------------------
Example 1

PREDICTION: वह कहता है कि मुख्य राजनीतिक दलों का कहना है कि आज का समय नहीं बढाया जाना चाहिए और कल के मंत्री का जवाब दिया जाना चाहिए। :＼

The minister said that he will reply tomorrow. :＼

मंत्री ने कहा कि कल का जवाब दिया जाएगा। :＼

The minister
LABEL:      उन्होंने यह भी कहा- ज्यादातर राजनीतिक पार्टियां ‘कह रही हैं कि वक्त आज नहीं बढ़ाया जाना चाहिए और मंत्री कल जवाब दे सकते हैं…।


Generating Outputs:   3%|▎         | 2/60 [00:07<03:26,  3.56s/it]


---------------------------------
Example 2

PREDICTION: क्या है, भारत में... #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+#
LABEL:      दरअसल, भारतीय . पढ़ें


...


---------------------------------
Example 58

PREDICTION: जुलाई से सितम्बर के तिमाह में GPF ने 7.6% की दर से लाभदायक रहा।.…

The following is a list of the top 10 highest grossing films of 2010..…

The following is a list of the top 10 highest grossing films of 2010.…

The following is a list of the top 10 highest grossing films of 2010.…

The following is a list
LABEL:      जुलाई-सितंबर तिमाही में जीपीएफ पर ब्‍याज 7.6 प्रतिशत वार्षिक था।


Generating Outputs:  98%|█████████▊| 59/60 [03:10<00:03,  3.10s/it]


---------------------------------
Example 59

PREDICTION: परिवार को यह सीखने के बाद कि हैरी और मेगन के लिए पिछले कुछ वर्षों के लिए कितना मुश्किल है, वह कहा कि "महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महाराज"महार
LABEL:      बकिंघम पैलेस के बयान में कहा गया है कि पूरा परिवार यह जानकर दुखी है कि हैरी और मेगन के लिए पिछले कुछ वर्ष चुनौतीपूर्ण रहे हैं।


Generating Outputs: 100%|██████████| 60/60 [03:14<00:00,  3.24s/it]


---------------------------------
Example 60

PREDICTION: पुलिस अधिकारी ने मृतकों के परिवार के साथ संपर्क की कोशिश कर रहे हैं। #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+# #+#
LABEL:      वहीँ पुलिस मुर्तक और घायल लोगो के परिवार से संपर्क करने की कोशिश कर रही है.


4.AMPLIFYING THE NEURONS

In [7]:
amplify_factor=1.5

In [8]:
# ==========================================================
#  Define inhibition hook for LLaMA-3.x MLPs
# ==========================================================
for activation_mask in activation_masks:
    if activation_mask is not None:

        def factory(mask):
            def llama_forward(self, x):
                gate = self.gate_proj(x)
                up = self.up_proj(x)
                activation = F.silu(gate)

                mask_tensor = torch.ones(activation.size(-1), device=activation.device,dtype=activation.dtype)
                mask_tensor[mask.to(activation.device).long()] =amplify_factor

                activation = activation * mask_tensor
                x = activation * up
                x = self.down_proj(x)
                return x
            return llama_forward

        # Inject hook layer-wise
        for i, layer_mask in enumerate(activation_mask):
            
            obj = model.model.layers[i].mlp
            obj.forward = MethodType(factory(layer_mask.to(device).long()), obj)

            

print("✅ Mask hooks injected.")


✅ Mask hooks injected.


In [9]:
from tqdm import tqdm
preds=[]
print("\n--- Model Outputs ---")

# Find a good stop token ID. For Llama 3, <|end_of_turn|> is best.
try:
    stop_token_id = tokenizer.convert_tokens_to_ids("<|end_of_turn|>")
    if stop_token_id == tokenizer.unk_token_id:
        stop_token_id = tokenizer.eos_token_id
except:
    stop_token_id = tokenizer.eos_token_id
print(f"Using stop token ID: {stop_token_id}")

for i in tqdm(range(len(test_token)), desc="Generating Outputs"):
    input_ids = torch.tensor([test_token[i]], device=device)
    attention_mask = torch.ones_like(input_ids) # <-- FIX 1: Add attention mask
    label = labels[i].strip()

    # Generate output
    with torch.no_grad(): # Ensure no gradients are computed
        output_ids = model.generate(
            input_ids,
            attention_mask=attention_mask, # <-- FIX 2: Pass attention mask
            max_new_tokens=100,
            do_sample=False, # <-- FIX 3: Add this for greedy decoding
            # temperature=0.0, # No longer needed when do_sample=False
            # top_p=1.0,       # No longer needed when do_sample=False
            eos_token_id=stop_token_id,
            pad_token_id=tokenizer.eos_token_id # Set pad_token_id
        )
    
    # Decode the *new* tokens only
    new_tokens = output_ids[0][len(test_token[i]):]
    prediction = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # --- 7. Display Results ---
    preds.append(prediction)
    print("\n---------------------------------")
    print(f"Example {i+1}")
    # prompt_text = tokenizer.decode(test_token[i], skip_special_tokens=True)
    # print(f"\nPROMPT:\n{prompt_text}") # Uncomment this line if you want to see the full prompt
    print(f"\nPREDICTION: {prediction}")
    print(f"LABEL:      {label}")


--- Model Outputs ---
Using stop token ID: 128001


Generating Outputs:   2%|▏         | 1/60 [00:04<04:20,  4.42s/it]


---------------------------------
Example 1

PREDICTION: वे कहते हैं कि समय को आज और कल के लिए नहीं बढ़ाया जाना चाहिए और मंत्री को कल का जवाब देना चाहिए.…


The government is not a good one. นางs are not a good one.…


The government is not a good one.…


The government is not a good one.…


The government is not a good one.…


The government is not a good
LABEL:      उन्होंने यह भी कहा- ज्यादातर राजनीतिक पार्टियां ‘कह रही हैं कि वक्त आज नहीं बढ़ाया जाना चाहिए और मंत्री कल जवाब दे सकते हैं…।


Generating Outputs:   3%|▎         | 2/60 [00:08<03:55,  4.06s/it]


---------------------------------
Example 2

PREDICTION: भला भारत के साथ...]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏
LABEL:      दरअसल, भारतीय . पढ़ें


...


---------------------------------
Example 58

PREDICTION: जुलाई-सितंबर के क्वार्टर में ग्रुप फॉर फाइनेंस प्रोवाइडेड 7.6% की दर का वापसी दी।.…

The company has a good reputation. นาง.二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二二
LABEL:      जुलाई-सितंबर तिमाही में जीपीएफ पर ब्‍याज 7.6 प्रतिशत वार्षिक था।


Generating Outputs:  98%|█████████▊| 59/60 [03:36<00:03,  3.49s/it]


---------------------------------
Example 59

PREDICTION: हैरी और मेगन के लिए पिछले कुछ वर्षों का पूरा प्रभाव कितना कठिन है, यह जानकर सभी परिवार को दुखी हो गया है.]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏]‏
LABEL:      बकिंघम पैलेस के बयान में कहा गया है कि पूरा परिवार यह जानकर दुखी है कि हैरी और मेगन के लिए पिछले कुछ वर्ष चुनौतीपूर्ण रहे हैं।


Generating Outputs: 100%|██████████| 60/60 [03:40<00:00,  3.67s/it]


---------------------------------
Example 60

PREDICTION: पुलिस के प्रयति के तहत मृतकों के परिवारों के साथ संपर्टि कर रहे हैं। นางuser]‏

The police are trying to contact the family members of the deceased and the injured. นางuser]‏

The police are trying to contact the family members of the deceased and the injured. นางuser]‏

The police are trying to contact the family members of the deceased and the injured. นางuser]‏

The
LABEL:      वहीँ पुलिस मुर्तक और घायल लोगो के परिवार से संपर्क करने की कोशिश कर रही है.


In [10]:
from bert_score import score

# Compute BERTScore for all predictions vs labels
P, R, F1 = score(preds, labels, lang="hi")

# Print individual F1 scores
print("\n--- BERTScore for each example ---")
for i, f in enumerate(F1):
    print(f"Example {i+1}: {f.item():.4f}")

# Print average score
print("\nAverage BERTScore F1:", F1.mean().item())



--- BERTScore for each example ---
Example 1: 0.6711
Example 2: 0.5038
Example 3: 0.6855
Example 4: 0.8093
Example 5: 0.5856
Example 6: 0.7328
Example 7: 0.6639
Example 8: 0.5550
Example 9: 0.5554
Example 10: 0.6336
Example 11: 0.6189
Example 12: 0.6049
Example 13: 0.6366
Example 14: 0.7064
Example 15: 0.7186
Example 16: 0.6245
Example 17: 0.7225
Example 18: 0.6573
Example 19: 0.6191
Example 20: 0.6059
Example 21: 0.6399
Example 22: 0.8056
Example 23: 0.6210
Example 24: 0.7121
Example 25: 0.5567
Example 26: 0.6613
Example 27: 0.6418
Example 28: 0.7235
Example 29: 0.5185
Example 30: 0.7601
Example 31: 0.6889
Example 32: 0.8269
Example 33: 0.7368
Example 34: 0.7289
Example 35: 0.7617
Example 36: 0.7191
Example 37: 0.6134
Example 38: 0.6187
Example 39: 0.7786
Example 40: 0.6705
Example 41: 0.5913
Example 42: 0.6563
Example 43: 0.5516
Example 44: 0.6977
Example 45: 0.7596
Example 46: 0.6731
Example 47: 0.5906
Example 48: 0.7274
Example 49: 0.5796
Example 50: 0.5240
Example 51: 0.6596
Exam